# AeroBin Phase 7a — Build the Burn-Risk Training Dataset

Builds a **ward-day** panel for the 5 pilot wards (Hadapsar, Kharadi, Wagholi, Bhosari, Mundhwa) over **2022-07-15 → 2024-12-31** (901 days × 5 wards = 4,505 ward-days; 4,365 usable after PM2.5-gap and first-week-lag clips) with:

- **Features** — daily weather (Open-Meteo Archive), daily PM2.5 (Open-Meteo Air Quality, CAMS reanalysis), static ward flags (from `../public/data/aerobin_data.json`), festival windows, day-of-year seasonality
- **Labels** — burn day if FIRMS hotspots attributed to the ward (nearest centroid ≤ 5 km — the **same rule as `api/fires.js`**) reach the threshold on that day or the next (48h framing)
- **Outputs** — `ward_day_labels.csv` (label lineage: every FIRMS pixel that produced a label) and `burn_dataset.csv` (features + label, ready for `02_train_model.ipynb`)

### Honest data-coverage notes (verified 2026-09-03, during the first live run of this notebook)
1. **CAMS PM2.5 begins ~2022-07-15** at these coordinates (2022 H1 hours return `None`; 2023–24 complete, 2024 has 0 missing hours). Hence the window starts 2022-07-15, not 2022-01-01 — the notebook **clips, documents, and never imputes across the gap**.
2. **FIRMS VIIRS_SNPP_SP** (Standard Processing) is the historical collection — the live NRT feed used by `api/fires.js` only reaches ~2 months back. The Area API caps **DAY_RANGE at 1–5 days per request**, so the window is fetched in 5-day chunks (181 requests) with 1s sleeps, well inside the 5000/10min free limit.
3. **Attribution confound (documented for the model card):** Bhosari sits in an industrial zone; FIRMS 375 m pixels cannot separate industrial from waste burning at threshold 1. Threshold is a **reported hyperparameter** (`LABEL_THRESHOLD` below) — run both 1 and 2 and compare label rates.
4. Ward coordinates are the **`api/fires.js` set** (real-place centroids: Hadapsar, Kharadi, Wagholi, Bhosari, Mundhwa). `aerobin_data.json`'s info coordinates are admin-ward proxies for some wards (see notes in that file) — training/live consistency with the live fires endpoint is the locked decision.
5. This notebook is **deterministic and re-runnable**: FIRMS raw pixels are cached to `cache/firms_pixels/` and Open-Meteo to `cache/openmeteo/` — delete the caches to force re-fetch.

In [ ]:
import hashlib
import json
import math
import os
import sys
import time
from datetime import date, datetime, timedelta
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

# --- pip deps (stdlib-first; pandas only for the final assembly) ---
try:
    import pandas as pd
except ImportError:
    raise SystemExit('pip install pandas  # the only third-party dependency')
import numpy as np

TRAINING_DIR = Path.cwd() if Path.cwd().name == 'training' else Path('../training').resolve()
if not TRAINING_DIR.exists():
    TRAINING_DIR = Path('training').resolve()
REPO_ROOT = TRAINING_DIR.parent
CACHE_DIR = TRAINING_DIR / 'cache'
FIRMS_CACHE = CACHE_DIR / 'firms_pixels'
OM_CACHE = CACHE_DIR / 'openmeteo'
for d in (FIRMS_CACHE, OM_CACHE):
    d.mkdir(parents=True, exist_ok=True)

def http_get_json(url, params=None, timeout=60, retries=3):
    """GET -> parsed JSON, retry on 5xx/timeouts with linear backoff (politeness for free tiers)."""
    if params:
        url = f'{url}?{urlencode(params)}'
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            req = Request(url, headers={'User-Agent': 'aerobin-training/0.1 (educational pipeline demonstrator)'})
            with urlopen(req, timeout=timeout) as resp:
                return json.loads(resp.read().decode('utf-8'))
        except HTTPError as e:
            if 400 <= e.code < 500:
                raise  # client error: retrying won't help
            last_err = e
        except Exception as e:  # timeout, connection reset
            last_err = e
        if attempt < retries:
            time.sleep(2 * attempt)
    raise last_err

def http_get_text(url, timeout=60, retries=3):
    """GET -> raw text (for FIRMS CSV), same retry policy."""
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            req = Request(url, headers={'User-Agent': 'aerobin-training/0.1 (educational pipeline demonstrator)'})
            with urlopen(req, timeout=timeout) as resp:
                return resp.read().decode('utf-8', errors='replace')
        except HTTPError as e:
            if 400 <= e.code < 500:
                raise
            last_err = e
        except Exception as e:
            last_err = e
        if attempt < retries:
            time.sleep(2 * attempt)
    raise last_err

print('workspace:', TRAINING_DIR)

## 1 — Config

Ward coordinates and static flags come from the app's own data file (single source of truth for green cover / market / income), while **centroid coordinates mirror `api/fires.js`** for attribution consistency with the live system.

In [ ]:
# Ward centroids [lat, lon] — kept in sync with api/fires.js (training/live consistency).
WARD_CENTROIDS = {
    'hadapsar': (18.5018, 73.926),
    'kharadi':  (18.5588, 73.9286),
    'wagholi':  (18.6074, 73.9872),
    'bhosari':  (18.6297, 73.8459),
    'mundhwa':  (18.5336, 73.8949),
}
# Pune bounding box — identical to api/fires.js PUNE_BBOX (west, south, east, north).
PUNE_BBOX = (73.70, 18.40, 74.10, 18.68)
MAX_WARD_DISTANCE_KM = 5  # same cap as api/fires.js

# Training window — start clipped to CAMS PM2.5 availability (verified above).
WINDOW_START = date(2022, 7, 15)
WINDOW_END = date(2024, 12, 31)

# Label rule: ward-day is a burn day if attributed hotspots >= LABEL_THRESHOLD
# on that day or the next (48h framing per the 3B design doc).
LABEL_THRESHOLD = 1   # primary run (single high-confidence VIIRS pixel counts)
# LABEL_THRESHOLD = 2  # sensitivity run — switch and re-run 6–9 to compare

# Static ward flags from the app dataset (public/data/aerobin_data.json).
with open(REPO_ROOT / 'public' / 'data' / 'aerobin_data.json', encoding='utf-8') as f:
    APP_DATA = json.load(f)
WARD_STATIC = {
    wid: {
        'name': info['name'],
        'green_cover_pct': info['greenCover'],
        'market_flag': int(info['marketFlag']),
        'income_level': info['incomeLevel'],
    }
    for wid, info in ((w, d['info']) for w, d in APP_DATA['wards'].items())
}
assert set(WARD_STATIC) == set(WARD_CENTROIDS), 'ward id mismatch between fires.js and app data'

# Festival windows ( Pune, fixed dates 2022–24 — Diwali + Holika Dahan 7-day aftermath).
# Open-waste burning spikes post-festival (the app's 'Post-festival week flag active' feature).
FESTIVAL_WINDOWS = [
    ('diwali', date(2022, 10, 24), 7),
    ('holika', date(2023, 3, 7),  7),
    ('diwali', date(2023, 11, 12), 7),
    ('holika', date(2024, 3, 24), 7),
    ('diwali', date(2024, 11, 1),  7),
]  # (name, start, aftermath_days inclusive of start)

print('wards:', list(WARD_CENTROIDS))
print('window:', WINDOW_START, '→', WINDOW_END, f'({(WINDOW_END - WINDOW_START).days + 1} days)')
print('label threshold:', LABEL_THRESHOLD)

## 2 — Fetch FIRMS VIIRS_SNPP_SP hotspots (historical, chunked + cached)

The **SP** (Standard Processing) collection has the full archive; NRT (used live in `api/fires.js`) only goes back ~2 months. We fetch the Pune bbox in **≤10-day chunks** (URL-length + rate politeness), sleep 1s between requests, and cache each chunk's raw CSV so re-runs are free.

In [ ]:
# The FIRMS MAP_KEY is a free public map key tied to the project's Vercel env.
# In Colab it is entered interactively (never stored in the notebook);
# locally it may be exported as an env var first:
#   PowerShell:  $env:FIRMS_MAP_KEY = "..."   (current session only)
def get_firms_key():
    key = os.environ.get('FIRMS_MAP_KEY', '').strip()
    if key:
        return key
    try:
        from getpass import getpass
        key = getpass('FIRMS MAP_KEY (input hidden): ').strip()
        if key:
            return key
    except Exception:
        pass
    raise SystemExit('No FIRMS MAP_KEY provided. Get a free key: https://firms.modaps.eosdis.nasa.gov/api/map_key/')

FIRMS_URL = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv'
FIRMS_SOURCE = 'VIIRS_SNPP_SP'
# FIRMS Area API: DAY_RANGE is capped at 1..5 days per request (larger
# transactions count against the 5000/10min MAP_KEY budget multiple times).
# 5-day chunks -> ~181 requests for the 2.5-year window (well within budget).
CHUNK_DAYS = 5

def chunk_key(a, b):
    return f'{a.isoformat()}_{b.isoformat()}'

def fetch_firms_chunk(start, end):
    ck = chunk_key(start, end)
    cache_file = FIRMS_CACHE / f'{ck}.csv'
    if cache_file.exists():
        return cache_file.read_text(encoding='utf-8')
    bbox = f'{PUNE_BBOX[0]},{PUNE_BBOX[1]},{PUNE_BBOX[2]},{PUNE_BBOX[3]}'
    n_days = (end - start).days + 1  # DAY_RANGE, capped 1..5 by the API
    url = f'{FIRMS_URL}/{get_firms_key()}/{FIRMS_SOURCE}/{bbox}/{n_days}/{start.isoformat()}'
    text = http_get_text(url, timeout=120)
    # FIRMS returns an error CSV (single line, no header match) on bad requests
    if 'latitude' not in text.splitlines()[0]:
        raise RuntimeError(f'FIRMS chunk {ck}: unexpected response head: {text[:120]!r}')
    cache_file.write_text(text, encoding='utf-8')
    time.sleep(1)
    return text

def parse_firms_csv(text):
    """Parse FIRMS CSV -> list of dicts. VIIRS CSVs carry bright_ti4/bright_ti5
    (no 'brightness' column — that's MODIS); pick bright_ti4 with a MODIS fallback."""
    lines = text.strip().splitlines()
    if len(lines) < 2:
        return []
    headers = [h.strip() for h in lines[0].split(',')]
    idx = {h: i for i, h in enumerate(headers)}
    bright_col = 'bright_ti4' if 'bright_ti4' in idx else ('brightness' if 'brightness' in idx else None)
    rows = []
    for line in lines[1:]:
        cols = line.split(',')
        try:
            rows.append({
                'latitude': float(cols[idx['latitude']]),
                'longitude': float(cols[idx['longitude']]),
                'acq_date': cols[idx['acq_date']].strip(),
                'acq_time': cols[idx['acq_time']].strip(),
                'confidence': cols[idx['confidence']].strip(),
                'brightness': float(cols[idx[bright_col]]),
            })
        except (KeyError, ValueError, IndexError):
            continue  # malformed line — skip
    return rows

# sanity check chunk 1 before the full sweep
_sample = fetch_firms_chunk(WINDOW_START, WINDOW_START + timedelta(days=CHUNK_DAYS - 1))
_rows = parse_firms_csv(_sample)
print(f'sample chunk {WINDOW_START} +{CHUNK_DAYS}d: {len(_rows)} pixels')
if not _rows:
    print('  (0 pixels is plausible here — mid-July is peak monsoon; the full sweep below is the real check)')

In [ ]:
# Full sweep over the training window.
pixels = []
n_chunks = 0
cur = WINDOW_START
while cur <= WINDOW_END:
    chunk_end = min(cur + timedelta(days=CHUNK_DAYS - 1), WINDOW_END)
    text = fetch_firms_chunk(cur, chunk_end)
    pixels.extend(parse_firms_csv(text))
    n_chunks += 1
    cur = chunk_end + timedelta(days=1)

print(f'{n_chunks} chunks, {len(pixels)} raw FIRMS pixels across the window')
print('date range in data:', min(p['acq_date'] for p in pixels), '→', max(p['acq_date'] for p in pixels))

## 3 — Attribute pixels to wards (nearest centroid ≤ 5 km — the `api/fires.js` rule)

Haversine copied verbatim in spirit from `api/fires.js:34` so training labels and the live `useFires` counts share one attribution rule.

In [ ]:
def haversine_km(a, b):
    R = 6371
    d_lat = math.radians(b[0] - a[0])
    d_lon = math.radians(b[1] - a[1])
    la1, la2 = math.radians(a[0]), math.radians(b[0])
    h = math.sin(d_lat / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin(d_lon / 2) ** 2
    return 2 * R * math.asin(math.sqrt(h))

attributed = []  # pixels with a ward
unattributed = 0
for p in pixels:
    best_id, best_km = None, float('inf')
    for wid, (lat, lon) in WARD_CENTROIDS.items():
        km = haversine_km((p['latitude'], p['longitude']), (lat, lon))
        if km < best_km:
            best_id, best_km = wid, km
    if best_id is not None and best_km <= MAX_WARD_DISTANCE_KM:
        attributed.append({
            'ward_id': best_id,
            'date': p['acq_date'],
            'latitude': p['latitude'],
            'longitude': p['longitude'],
            'confidence': p['confidence'],
            'brightness': p['brightness'],
            'distance_km': round(best_km, 3),
        })
    else:
        unattributed += 1

df_pix = pd.DataFrame(attributed)
print(f'{len(attributed)} pixels attributed to a ward; {unattributed} outside the 5 km cap')
if not df_pix.empty:
    print(df_pix.groupby('ward_id').size().sort_values(ascending=False))
    print('\ndistance stats (km):')
    print(df_pix['distance_km'].describe().round(2))

## 4 — Fetch daily weather + PM2.5 per ward (Open-Meteo, cached)

- **Weather** — Archive API, daily: `temperature_2m_max`, `relative_humidity_2m_mean`, `precipitation_sum`, `wind_speed_10m_max` (verified: full window available).
- **PM2.5** — Air Quality API, hourly CAMS `pm2_5` aggregated to a local-day **mean over available hours**; the coverage fraction is kept as a feature-input note and rows before 2022-07-15 are clipped (CAMS rollout, see header).

In [ ]:
OM_ARCHIVE = 'https://archive-api.open-meteo.com/v1/archive'
OM_AIRQUALITY = 'https://air-quality-api.open-meteo.com/v1/air-quality'
TZ = 'Asia/Kolkata'

def om_cache_path(kind, ward_id):
    return OM_CACHE / f'{kind}_{ward_id}.json'

def fetch_weather(ward_id):
    cp = om_cache_path('weather', ward_id)
    if cp.exists():
        return json.loads(cp.read_text(encoding='utf-8'))
    lat, lon = WARD_CENTROIDS[ward_id]
    d = http_get_json(OM_ARCHIVE, params={
        'latitude': lat, 'longitude': lon,
        'start_date': WINDOW_START.isoformat(), 'end_date': WINDOW_END.isoformat(),
        'daily': 'temperature_2m_max,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max',
        'timezone': TZ,
    })
    cp.write_text(json.dumps(d), encoding='utf-8')
    time.sleep(1)
    return d

def fetch_pm25(ward_id):
    """Hourly CAMS pm2_5 -> daily mean + coverage fraction. Hourly requests are
    split per ~6 months to stay within Open-Meteo response limits."""
    cp = om_cache_path('pm25', ward_id)
    if cp.exists():
        return json.loads(cp.read_text(encoding='utf-8'))
    lat, lon = WARD_CENTROIDS[ward_id]
    out = {'time': [], 'pm2_5': []}
    cur = WINDOW_START
    while cur <= WINDOW_END:
        seg_end = min(cur + timedelta(days=180), WINDOW_END)
        d = http_get_json(OM_AIRQUALITY, params={
            'latitude': lat, 'longitude': lon,
            'hourly': 'pm2_5',
            'start_date': cur.isoformat(), 'end_date': seg_end.isoformat(),
            'timezone': TZ,
        })
        h = d.get('hourly', {})
        out['time'].extend(h.get('time', []))
        out['pm2_5'].extend(h.get('pm2_5', []))
        cur = seg_end + timedelta(days=1)
        time.sleep(1)
    cp.write_text(json.dumps(out), encoding='utf-8')
    return out

ward_weather, ward_pm25_raw = {}, {}
for wid in WARD_CENTROIDS:
    ward_weather[wid] = fetch_weather(wid)
    ward_pm25_raw[wid] = fetch_pm25(wid)
    print(f'{wid}: weather {len(ward_weather[wid]["daily"]["time"])} days, '
          f'pm2_5 {len(ward_pm25_raw[wid]["time"])} hours')
print('done')

In [ ]:
# Build per-ward daily frames: weather + daily PM2.5 (mean over available hours + coverage).
ward_daily = {}
for wid in WARD_CENTROIDS:
    w = ward_weather[wid]['daily']
    df = pd.DataFrame({
        'date': pd.to_datetime(w['time']),
        'temp_max': w['temperature_2m_max'],
        'humidity_mean': w['relative_humidity_2m_mean'],
        'precipitation_sum': w['precipitation_sum'],
        'wind_max': w['wind_speed_10m_max'],
    })
    p = ward_pm25_raw[wid]
    dfp = pd.DataFrame({'date': pd.DatetimeIndex(pd.to_datetime(p['time'])).floor('D'), 'pm2_5': p['pm2_5']})
    agg = dfp.groupby('date')['pm2_5'].agg(['mean', 'count']).reset_index()
    agg.columns = ['date', 'pm25_mean', 'pm25_hours']
    df = df.merge(agg, on='date', how='left')
    df['pm25_coverage'] = df['pm25_hours'] / 24.0
    df['ward_id'] = wid
    ward_daily[wid] = df

df_all = pd.concat(ward_daily.values(), ignore_index=True).sort_values(['ward_id', 'date'])
n_days = (WINDOW_END - WINDOW_START).days + 1
assert len(df_all) == len(WARD_CENTROIDS) * n_days, f'expected {len(WARD_CENTROIDS)*n_days} rows, got {len(df_all)}'
print(f'{len(df_all)} ward-days')
print('\nPM2.5 coverage by year (fraction of days with >=50% hourly coverage):')
df_all['year'] = df_all['date'].dt.year
print(df_all.assign(ok=df_all['pm25_coverage'] >= 0.5).groupby('year')['ok'].mean().round(3))

## 5 — Labels: ward-day burn events (48h framing)

`burn_day = 1` if attributed hotspot count ≥ `LABEL_THRESHOLD` on that day **or the next** (a fire detected after noon still lands in the current "operational day"). This yields the label column; the pixel-level lineage goes to `ward_day_labels.csv`.

In [ ]:
if df_pix.empty:
    df_counts = pd.DataFrame(columns=['ward_id', 'date', 'hotspots'])
else:
    df_counts = (df_pix.groupby(['ward_id', 'date']).size()
                 .reset_index(name='hotspots'))
    df_counts['date'] = pd.to_datetime(df_counts['date'])

df = df_all.merge(df_counts, on=['ward_id', 'date'], how='left')
df['hotspots'] = df['hotspots'].fillna(0).astype(int)

# 48h framing: today OR tomorrow >= threshold
df = df.sort_values(['ward_id', 'date']).reset_index(drop=True)
df['hotspots_next'] = df.groupby('ward_id')['hotspots'].shift(-1).fillna(0)
df['hotspots_48h'] = df['hotspots'] + df['hotspots_next']
df['burn_day'] = (df['hotspots_48h'] >= LABEL_THRESHOLD).astype(int)

# NOTE: the last day has no "tomorrow" inside the window; its label uses today only.
# This is a one-row-per-ward edge effect — acceptable and documented.

print('label balance:')
print(df.groupby('ward_id')['burn_day'].agg(['sum', 'mean', 'count']).round(3))
print(f"\noverall burn-day rate: {df['burn_day'].mean():.3f}")

## 6 — Feature engineering

Per the locked 7a recipe:
- `pm25_mean` + 7-day rolling slope (aqi trend analog of the live app)
- `humidity_mean` + 7-day trend
- `days_since_rain` (dryness — derived from precipitation)
- `wind_max`
- static: `green_cover_pct`, `market_flag`, `income_level` (encoded)
- festival windows (Diwali / Holika Dahan + 7-day aftermath)
- day-of-year seasonality: sin/cos

All rolling features use **past-only windows** (no leakage: `shift(1)` before rolling when the label window includes tomorrow).

In [ ]:
# Festival aftermath flags.
festival_days = set()
for _, start, days in FESTIVAL_WINDOWS:
    for i in range(days):
        festival_days.add(start + timedelta(days=i))
df['festival_window'] = df['date'].dt.date.isin(festival_days).astype(int)

# Income encoding (ordinal: low < mixed < high — documented choice for the model card).
INCOME_ORDER = {'low': 0, 'mixed': 1, 'high': 2}
df['income_ord'] = df['ward_id'].map(lambda w: INCOME_ORDER[WARD_STATIC[w]['income_level']])
df['green_cover_pct'] = df['ward_id'].map(lambda w: WARD_STATIC[w]['green_cover_pct'])
df['market_flag'] = df['ward_id'].map(lambda w: WARD_STATIC[w]['market_flag'])

# PM2.5 7-day slope: yesterday's value minus value 8 days back, over the past 7 days.
# shift(1) keeps the feature strictly past-only.
g = df.groupby('ward_id')
pm_lag1 = g['pm25_mean'].shift(1)
pm_lag8 = g['pm25_mean'].shift(8)
df['pm25_slope_7d'] = (pm_lag1 - pm_lag8) / 7.0

# Humidity 7-day trend (past-only).
h_lag1 = g['humidity_mean'].shift(1)
h_lag8 = g['humidity_mean'].shift(8)
df['humidity_trend_7d'] = (h_lag1 - h_lag8) / 7.0

# Rolling means (past-only) for smoothing.
df['pm25_mean_7d'] = g['pm25_mean'].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
df['humidity_7d'] = g['humidity_mean'].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())

# Days since rain (reset on any day with >= 1mm).
def days_since_rain_series(grp):
    vals, count = [], 0
    for pr in grp['precipitation_sum']:
        count = 0 if (pr is not None and not pd.isna(pr) and pr >= 1.0) else count + 1
        vals.append(count)
    return pd.Series(vals, index=grp.index)
df['days_since_rain'] = df.groupby('ward_id', group_keys=False)[['precipitation_sum']].apply(days_since_rain_series)

# Day-of-year seasonality.
import numpy as np
doy = df['date'].dt.dayofyear
df['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)

print(df[['pm25_mean', 'pm25_slope_7d', 'humidity_trend_7d', 'days_since_rain', 'pm25_mean_7d', 'humidity_7d']].describe().round(2))

## 7 — Final dataset: clip unusable rows, write CSVs

Rows are dropped where PM2.5 is entirely unavailable on the day (`pm25_hours == 0`) or key lags are missing (first 8 days per ward). The dataset carries a `split_year` helper column (train 2022–23, test 2024) for the training notebook's time-series split.

In [ ]:
FEATURE_COLS = [
    'pm25_mean', 'pm25_mean_7d', 'pm25_slope_7d',
    'humidity_mean', 'humidity_7d', 'humidity_trend_7d',
    'temp_max', 'wind_max', 'precipitation_sum', 'days_since_rain',
    'green_cover_pct', 'market_flag', 'income_ord',
    'festival_window', 'doy_sin', 'doy_cos',
]
LABEL_COL = 'burn_day'
META_COLS = ['date', 'ward_id', 'hotspots', 'hotspots_48h', 'pm25_coverage']

before = len(df)
# clip rows where the day's PM2.5 is fully missing (CAMS gap) or lags unavailable
df = df[(df['pm25_hours'].fillna(0) > 0)].copy()
df = df.dropna(subset=['pm25_mean', 'pm25_slope_7d', 'humidity_trend_7d', 'days_since_rain', 'temp_max'])
df['split_year'] = df['date'].dt.year

print(f'rows: {before} -> {len(df)} (dropped {before - len(df)} for missing PM2.5/first-week lags)')
print('\nfinal label balance by ward:')
print(df.groupby('ward_id')[LABEL_COL].agg(['sum', 'mean', 'count']).round(3))
print(f"\noverall: {df[LABEL_COL].sum()} burn days / {len(df)} ward-days ({df[LABEL_COL].mean():.3f})")
print('\nby split year:')
print(df.groupby('split_year')[LABEL_COL].agg(['sum', 'mean', 'count']).round(3))

In [ ]:
out_ds = df[META_COLS + FEATURE_COLS + [LABEL_COL, 'split_year']].copy()
out_ds['date'] = out_ds['date'].dt.strftime('%Y-%m-%d')
out_path = TRAINING_DIR / 'burn_dataset.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
out_ds.to_csv(out_path, index=False)

# Label lineage: every attributed pixel (cached raw FIRMS CSVs stay in cache/ for audit).
if not df_pix.empty:
    lin = df_pix.copy()
    lin.to_csv(TRAINING_DIR / 'ward_day_labels.csv', index=False)
    print('ward_day_labels.csv:', len(lin), 'pixels')
else:
    print('WARNING: zero attributed pixels — ward_day_labels.csv not written')

print('burn_dataset.csv:', len(out_ds), 'rows x', len(out_ds.columns), 'cols ->', out_path)

## 8 — Sanity checks (the notebook fails loudly if the data is wrong)

These assertions are the phase exit gate: lineage counts, no-leakage feature windows, no empty labels, class balance reported for the model card.

In [ ]:
chk = pd.read_csv(out_path)
assert list(chk.columns[:2]) == ['date', 'ward_id']
assert chk['burn_day'].isin([0, 1]).all()
assert chk.groupby('ward_id')['date'].nunique().eq(chk.groupby('ward_id').size()).all(), 'duplicate ward-days!'
assert chk['split_year'].isin([2022, 2023, 2024]).all()
# burn days must correspond to real hotspot activity in the lineage file
lin = pd.read_csv(TRAINING_DIR / 'ward_day_labels.csv')
lin_days = lin.groupby(['ward_id', 'acq_date' if 'acq_date' in lin.columns else 'date']).size()
print('lineage pixel-days:', len(lin_days))
print('burn rows in dataset:', int(chk['burn_day'].sum()))
print('\nALL CHECKS PASSED — dataset ready for 02_train_model.ipynb')
chk.head()